In [8]:
import collections
import math
import sys # For printing warnings
import csv
import bisect

class FootprintMRC:
    """
    A class to calculate the Miss-Ratio Curve (MRC) based on Footprint Theory
    for a streaming sequence of memory access keys. It also implements a
    dynamic programming algorithm for optimal slab reallocation.
    """

    # Define the standard slab size in bytes (used in query_mrc for internal calculations)
    SLAB_SIZE = 4 * 1024 * 1024 # 4 MB

    def __init__(self, k: int):
        """
        Initializes the MRC profiler with a circular buffer of size 'k'.

        Args:
            k (int): The maximum number of recent requests to keep in the
                     circular buffer for MRC calculation. Must be >= 1.
        """
        if k < 1:
            raise ValueError("Circular buffer size 'k' must be at least 1.")
        # Stores tuples of (key, class_id)
        self.circular_buffer = collections.deque(maxlen=k)

    def feed(self, key, class_id):
        """
        Feeds a new memory access request (key, class ID) into the circular buffer.
        If the buffer is full, the oldest entry is automatically discarded.

        Args:
            key: The unique identifier for the memory item being accessed.
            class_id: The identifier for the class this key belongs to.
        """
        self.circular_buffer.append((key, class_id))

    def _calculate_window_stats(self):
        """
        Calculates first_access_times, last_access_times, reuse_time_histogram,
        total_accesses (n), and unique_accesses (m) specifically for the
        current contents of the circular buffer, grouped by class_id.

        Returns:
            tuple: (first_access_times_by_class, last_access_times_by_class,
                    reuse_time_histogram_by_class, n_by_class, m_by_class)
            Where each returned value is a dictionary mapping class_id to its
            respective statistics for the current window.
        """
        first_access_times_by_class = collections.defaultdict(dict)
        last_access_times_by_class = collections.defaultdict(dict)
        reuse_time_histogram_by_class = collections.defaultdict(lambda: collections.defaultdict(int))
        n_by_class = collections.defaultdict(int)  # total accesses per class (also acts as local index)
        m_by_class = collections.defaultdict(int)  # unique accesses per class

        # Iterate through the circular buffer globally.
        # `global_idx` is the position within the circular buffer.
        for global_idx, (key, class_id) in enumerate(self.circular_buffer):
            # This `local_idx_for_current_access` is the effective position of this
            # access *within the sub-trace of its specific class*.
            local_idx_for_current_access = n_by_class[class_id] 

            # Increment the total count for this class. This will be the 'n' for this class's sub-trace.
            n_by_class[class_id] += 1

            # If this is the first time we've encountered this key within this specific class in *this window*
            if key not in first_access_times_by_class[class_id]:
                # Store the local index (not the global_idx)
                first_access_times_by_class[class_id][key] = local_idx_for_current_access
                m_by_class[class_id] += 1  # Increment unique access count for this class

            # If this key has been seen before within this class in *this window*, calculate its reuse time
            if key in last_access_times_by_class[class_id]:
                # Retrieve the previously stored local index
                prev_local_index = last_access_times_by_class[class_id][key]
                # Calculate reuse time using only local indices
                reuse_time = local_idx_for_current_access - prev_local_index
                reuse_time_histogram_by_class[class_id][reuse_time] += 1

            # Update the last access time for the current key within this class to its current local index
            last_access_times_by_class[class_id][key] = local_idx_for_current_access
        
        return (first_access_times_by_class, last_access_times_by_class,
                reuse_time_histogram_by_class, n_by_class, m_by_class)

    def _calculate_fp_values(self, first_access_times, last_access_times, reuse_time_histogram, n, m):
        """
        Calculates the footprint fp(w) for all possible window lengths 'w'
        from 0 up to the total number of accesses 'n' for a given class in the current window.
        This version takes the window-specific statistics for a single class as arguments.

        Args:
            first_access_times (dict): First access times for keys in the current window.
            last_access_times (dict): Last access times for keys in the current window.
            reuse_time_histogram (defaultdict): Reuse time counts for the current window.
            n (int): Total accesses in the current window.
            m (int): Unique accesses in the current window.

        Returns:
            dict: A dictionary where keys are window lengths (w) and values are
                  their corresponding average footprint fp(w). Returns an empty
                  dictionary if n is 0.
        """
        if n == 0:
            return {}

        fp_values = {0: 0.0}  # As per the definition fp(0) = 0 in the materials

        # Pre-computation for the R_w term (sum((t-w)*r_t))
        max_t = max(reuse_time_histogram.keys(), default=0)
        if n > 0:
            max_t = min(max_t, n - 1)

        sum_tr_suffix = [0.0] * (max_t + 2)
        sum_r_suffix = [0.0] * (max_t + 2)

        for t in range(max_t, 0, -1):
            current_r_t = reuse_time_histogram.get(t, 0)
            sum_tr_suffix[t] = sum_tr_suffix[t+1] + (t * current_r_t)
            sum_r_suffix[t] = sum_r_suffix[t+1] + current_r_t

        # Prepare 1-indexed f_i and l_i values for the formula
        f_values_1indexed = sorted([idx + 1 for idx in first_access_times.values()])
        l_values_1indexed = sorted([n - idx for idx in last_access_times.values()])

        # Initialize current sums and counts for F_w and L_w terms
        current_F_sum = sum(f_values_1indexed)
        current_F_count = len(f_values_1indexed)
        f_ptr = 0

        current_L_sum = sum(l_values_1indexed)
        current_L_count = len(l_values_1indexed)
        l_ptr = 0

        # Calculate fp(w) for each window length w from 1 to n
        for w in range(1, n + 1):
            # Calculate F_w = sum(max(0, f_i - w))
            while f_ptr < len(f_values_1indexed) and f_values_1indexed[f_ptr] <= w:
                current_F_sum -= f_values_1indexed[f_ptr]
                current_F_count -= 1
                f_ptr += 1
            F_w = current_F_sum - w * current_F_count

            # Calculate L_w = sum(max(0, l_i - w))
            while l_ptr < len(l_values_1indexed) and l_values_1indexed[l_ptr] <= w:
                current_L_sum -= l_values_1indexed[l_ptr]
                current_L_count -= 1
                l_ptr += 1
            L_w = current_L_sum - w * current_L_count

            R_w = 0.0
            if w + 1 <= max_t:
                R_w = sum_tr_suffix[w+1] - w * sum_r_suffix[w+1]
            
            denominator = n - w + 1
            if denominator <= 0:
                fp_values[w] = float(m)
                continue

            # Apply the main footprint formula - NO CAPPING to 0.0 here.
            # This is to match the "old correct" Python behavior where math.ceil
            # on negative values created a non-flat MRC.
            fp_val = float(m) - (1.0 / denominator) * (F_w + L_w + R_w)
            
            fp_values[w] = fp_val

        return fp_values

    def query_mrc(self, class_id_to_allocs_per_slab_map: dict, max_slab_cnt: int):
        """
        Calculates the Miss-Ratio Curve (MRC) based on the current requests
        in the circular buffer, expressed in terms of slab granularity,
        for each unique class_id. Also returns the mrc_delta for each class
        and the total access frequency for each class within the window.

        Args:
            class_id_to_allocs_per_slab_map (dict): A dictionary where keys are ClassId
                                                    and values are the number of objects
                                                    (of that class) that can be stored in one slab.
                                                    Must be > 0 for all relevant classes.
            max_slab_cnt (int): The maximum number of slabs to consider for the MRC.

        Returns:
            dict: A dictionary where keys are class_ids. Each value is a tuple:
                - mrc_points (dict): A dictionary where keys are slab counts and
                                     values are their corresponding miss ratios.
                - mrc_delta (dict): A dictionary where keys are slab counts (i)
                                    and values are mrc(i-1) - mrc(i).
                - access_frequency (int): The total number of requests for this class
                                          in the current circular buffer window.
        """
        (first_access_times_by_class, last_access_times_by_class,
         reuse_time_histogram_by_class, n_by_class, m_by_class) = self._calculate_window_stats()

        if not n_by_class:
            return {}

        if max_slab_cnt < 0:
            raise ValueError("max_slab_cnt cannot be negative.")

        all_class_mrc_results = {}

        for class_id in n_by_class.keys():
            n_window_for_class = n_by_class[class_id]
            m_window_for_class = m_by_class[class_id]
            first_access_times_for_class = first_access_times_by_class[class_id]
            last_access_times_for_class = last_access_times_by_class[class_id]
            reuse_time_histogram_for_class = reuse_time_histogram_by_class[class_id]

            if n_window_for_class == 0:
                # This case is technically handled by n_by_class.keys() only iterating existing classes,
                # but good for robustness if passed manually.
                all_class_mrc_results[class_id] = ({}, {}, 0)
                continue

            # Get allocs_per_slab for the current classId
            allocs_per_slab = class_id_to_allocs_per_slab_map.get(class_id)
            if allocs_per_slab is None or allocs_per_slab <= 0:
                raise ValueError(f"AllocsPerSlab for classId '{class_id}' not provided or not greater than 0.")

            # 1. Calculate all fp(w) values (in terms of number of objects) for the current class window
            fp_values_objects = self._calculate_fp_values(
                first_access_times_for_class, last_access_times_for_class,
                reuse_time_histogram_for_class, n_window_for_class, m_window_for_class
            )

            # 2. Create pairs of (slabs_needed, count_of_reuse_time_t)
            # for all reuse times that occurred in the current class window.
            fp_slab_pairs = []
            for t, r_t_count in reuse_time_histogram_for_class.items():
                if t in fp_values_objects: # Check if t (reuse time / window size) has a calculated footprint
                    fp_objects = fp_values_objects[t] # Footprint in terms of number of objects
                    
                    # Calculate the number of slabs required to hold this footprint (in objects)
                    # math.ceil handles negative numbers correctly (e.g., ceil(-0.5) is 0, ceil(-1.5) is -1).
                    slabs_needed = math.ceil(fp_objects / allocs_per_slab) if allocs_per_slab > 0 else float('inf')
                    fp_slab_pairs.append((slabs_needed, r_t_count))

            fp_slab_pairs.sort(key=lambda x: x[0])

            # 3. Construct the MRC in slab granularity for this class
            mrc_dict_class = collections.defaultdict(lambda: 1.0)
            mrc_dict_class[0] = 1.0  # Initial point: for 0 slabs, the miss ratio is 1.0

            cumulative_rt_sum = 0.0
            fp_slab_ptr = 0

            for slab_count in range(1, max_slab_cnt + 1):
                while fp_slab_ptr < len(fp_slab_pairs) and fp_slab_pairs[fp_slab_ptr][0] <= slab_count:
                    cumulative_rt_sum += fp_slab_pairs[fp_slab_ptr][1]
                    fp_slab_ptr += 1
                
                miss_ratio = 1.0 - (cumulative_rt_sum / n_window_for_class)
                mrc_dict_class[slab_count] = miss_ratio

            mrc_points_class = mrc_dict_class

            # 4. Calculate mrc_delta for this class
            mrc_delta_class = collections.defaultdict(lambda: 0.0)
            for i in range(1, max_slab_cnt + 1):
                prev_mrc_val = mrc_points_class.get(i - 1, mrc_points_class.get(max(mrc_points_class.keys(), default=0), 0.0))
                current_mrc_val = mrc_points_class.get(i, mrc_points_class.get(max(mrc_points_class.keys(), default=0), 0.0))
                mrc_delta_class[i] = prev_mrc_val - current_mrc_val
            
            all_class_mrc_results[class_id] = (mrc_points_class, mrc_delta_class, n_window_for_class)
        
        return all_class_mrc_results

    def reset_window_analysis(self):
        """
        Resets the circular buffer, effectively clearing all past requests
        and starting a new analysis window.
        """
        self.circular_buffer.clear()

    def _get_miss_ratio(self, mrc_points_dict, slab_count):
        """
        Helper to get the miss ratio for a given slab count from the MRC points.
        Assumes MRC points provide continuous data up to max_slab_cnt.
        """
        if slab_count == 0:
            return 1.0
        # If the exact slab_count is in the provided MRC points, use it.
        if slab_count in mrc_points_dict:
            return mrc_points_dict[slab_count]
        else:
            # If slab_count goes beyond what MRC has, assume miss ratio of the largest profiled count (or 0.0 if empty).
            # This is based on how query_mrc constructs its dict, providing continuous points.
            max_profiled_slab_count = max(mrc_points_dict.keys(), default=0)
            if slab_count > max_profiled_slab_count:
                return mrc_points_dict.get(max_profiled_slab_count, 0.0)
            # If slab_count is somehow less than minimum profiled (and not 0), it's an error in expectation
            # or an extremely small cache. Defaulting to 1.0 for these.
            return 1.0 

    def solve_slab_reallocation(self, class_id_to_allocs_per_slab_map: dict, current_slab_allocation: dict):
        """
        Solves the locality-aware memory allocation problem using dynamic programming.
        This algorithm aims to find an optimal distribution of a fixed total number of slabs
        across different size classes to minimize total cost (accesses * miss rate).

        Args:
            class_id_to_allocs_per_slab_map (dict): A dictionary mapping ClassId to the
                                                    number of objects (of that class) that
                                                    can be stored in one slab.
            current_slab_allocation (dict): A dictionary mapping ClassId to the current
                                            number of slabs allocated to that class.
                                            The sum of these slabs defines the total
                                            number of slabs to reallocate.

        Returns:
            tuple: A tuple containing:
                - mr_old (float): Total miss rate of the system with the current allocation.
                - mr_new (float): Total miss rate of the system with the new optimal allocation.
                - optimal_allocation (dict): A dictionary mapping ClassId to the new
                                             optimal number of slabs.
                - reassignment_plan (list): A list of (victim_class_id, receiver_class_id) tuples,
                                            indicating individual slab movements from old to new.
                                            Sorted by requests per current slab (ascending) for victims.
                - access_frequencies (dict): A dictionary mapping ClassId to the total number of
                                             requests for that class in the current window.
        """
        # The total number of slabs available for reallocation is derived from current_slab_allocation.
        max_total_slabs = sum(current_slab_allocation.values())

        # Max slabs for MRC profile for individual classes can be capped by max_total_slabs.
        # This defines the range for the dynamic programming 'j' variable as well.
        max_slabs_for_mrc_profile = max_total_slabs

        # 1. Query MRC data directly from the profiler instance (self)
        class_mrc_data = self.query_mrc(class_id_to_allocs_per_slab_map, max_slabs_for_mrc_profile)

        if not class_mrc_data:
            sys.stderr.write("Warning: No class MRC data available. Returning empty reallocation.\n")
            return 0.0, 0.0, {}, [], {}
        
        # Edge case: If no slabs to allocate and no objects profiled
        if max_total_slabs == 0 and not class_id_to_allocs_per_slab_map:
            return 0.0, 0.0, {}, [], {}

        # Get a consistent ordered list of class IDs for DP table indexing.
        class_ids = sorted(class_mrc_data.keys())
        num_classes = len(class_ids)

        # Store access_frequencies for return
        access_frequencies = {class_id: class_mrc_data[class_id][2] for class_id in class_ids}

        # Pre-calculate the cost table: cost_table[class_idx][slab_count]
        # cost_table[i][j] stores the cost (access_frequency * miss_ratio) for class_ids[i] with j slabs.
        cost_table = [[float('inf')] * (max_total_slabs + 1) for _ in range(num_classes)]

        for i, class_id in enumerate(class_ids):
            mrc_points, _, access_frequency = class_mrc_data[class_id]
            # Print for inspection
            sys.stdout.write(f"MRC points for class_id {class_id}: \n")
            slab_sample_points = [0, 1, 2, 3, 4, 10, 50, 100, 128] # Sample points to print
            for s in sorted(mrc_points.keys()):
                if s in slab_sample_points or (s % 20 == 0 and s > 0): # Print common samples or every 20th slab
                     sys.stdout.write(f"  {s} slab: {mrc_points[s]:.4f}\n")
            
            for j in range(max_total_slabs + 1):  # j is the number of slabs for this *single* class
                # Cap j to max_slabs_for_mrc_profile if it exceeds the profiling range for MRC lookup
                effective_j = min(j, max_slabs_for_mrc_profile) 
                miss_ratio = self._get_miss_ratio(mrc_points, effective_j) 
                cost_table[i][j] = access_frequency * miss_ratio

        # Dynamic Programming Tables
        # F[i][j]: min cost for first 'i' classes using 'j' slabs in total
        F = [[float('inf')] * (max_total_slabs + 1) for _ in range(num_classes + 1)]
        # B[i][j]: stores 'k', the number of slabs allocated to class_ids[i-1] to achieve F[i][j]
        B = [[0] * (max_total_slabs + 1) for _ in range(num_classes + 1)]

        # Base case: 0 classes, 0 slabs, cost is 0
        F[0][0] = 0.0

        # Fill DP tables
        for i in range(1, num_classes + 1):  # Current class index (1-based)
            for j in range(max_total_slabs + 1):  # Total slabs considered so far
                # Iterate through possible slabs 'k' to allocate to the current class (class_ids[i-1])
                for k in range(min(j, max_slabs_for_mrc_profile) + 1): # k cannot exceed max_slabs_for_mrc_profile
                    if F[i-1][j-k] != float('inf'):  # Check if the previous state was reachable
                        current_class_cost = cost_table[i-1][k]
                        temp_cost = F[i-1][j-k] + current_class_cost

                        if temp_cost < F[i][j]:
                            F[i][j] = temp_cost
                            B[i][j] = k

        # Reconstruct Optimal Allocation
        optimal_allocation = {}
        remaining_slabs = max_total_slabs
        for i in range(num_classes, 0, -1):  # Iterate from last class down to first (1-based index)
            class_id = class_ids[i-1]  # Get actual class_id (0-indexed list)
            slabs_for_this_class = B[i][remaining_slabs]
            optimal_allocation[class_id] = slabs_for_this_class
            remaining_slabs -= slabs_for_this_class
        
        # Ensure all classes are represented in optimal_allocation, even if 0 slabs
        for class_id in set(class_ids).union(current_slab_allocation.keys()):
            if class_id not in optimal_allocation:
                optimal_allocation[class_id] = 0

        # Calculate mr_old and mr_new (overall miss rates)
        total_requests_in_window = sum(access_frequencies.values())
        
        total_misses_old = 0.0
        for class_id, current_slabs in current_slab_allocation.items():
            if class_id in class_mrc_data:
                mrc_points, _, access_frequency = class_mrc_data[class_id]
                total_misses_old += access_frequency * self._get_miss_ratio(mrc_points, current_slabs)
            # If class_id not in class_mrc_data, its access_frequency is 0, so contribution is 0.

        total_misses_new = 0.0
        for class_id, optimal_slabs in optimal_allocation.items():
            if class_id in class_mrc_data:
                mrc_points, _, access_frequency = class_mrc_data[class_id]
                total_misses_new += access_frequency * self._get_miss_ratio(mrc_points, optimal_slabs)
            # If class_id not in class_mrc_data, its access_frequency is 0, so contribution is 0.

        mr_old = total_misses_old / total_requests_in_window if total_requests_in_window > 0 else 0.0
        mr_new = total_misses_new / total_requests_in_window if total_requests_in_window > 0 else 0.0

        # Determine Reassignment Plan
        reassignment_plan = []
        victim_slabs_to_move = []
        receiver_slabs_to_move = []

        # Iterate over all unique class_ids that were either current or optimal
        all_relevant_class_ids = sorted(set(current_slab_allocation.keys()).union(optimal_allocation.keys()))

        for class_id in all_relevant_class_ids:
            current_slabs = current_slab_allocation.get(class_id, 0)
            optimal_slabs = optimal_allocation.get(class_id, 0)

            if optimal_slabs < current_slabs:
                num_slabs_to_give = current_slabs - optimal_slabs
                victim_slabs_to_move.extend([class_id] * num_slabs_to_give)
            elif optimal_slabs > current_slabs:
                num_slabs_to_gain = optimal_slabs - current_slabs
                receiver_slabs_to_move.extend([class_id] * num_slabs_to_gain)
        
        # Sort victim_slabs_to_move by (access_frequency / current_slabs) in ascending order
        # Handle division by zero: if current_slabs is 0, it means the class is not currently
        # allocated any slabs, so it cannot be a victim providing slabs. It would have
        # float('inf') as its score, pushing it to the end (or not considered).
        victim_slabs_to_move.sort(key=lambda class_id: 
                                   (access_frequencies.get(class_id, 0) / current_slab_allocation.get(class_id, 1)) # Use 1 to avoid ZeroDivisionError if current_slabs becomes 0 (e.g. for an initial state of 0)
                                   if current_slab_allocation.get(class_id, 0) > 0 and access_frequencies.get(class_id, 0) > 0 else float('inf'))

        # Create individual (victim, receiver) pairs
        for i in range(min(len(victim_slabs_to_move), len(receiver_slabs_to_move))):
            reassignment_plan.append((victim_slabs_to_move[i], receiver_slabs_to_move[i]))

        return mr_old, mr_new, optimal_allocation, reassignment_plan, access_frequencies


In [9]:
trace_file_path = "/mydata/hongshu/traces/synth_static_202.csv"
max_rows = 1000_000

def process_file(trace_file_path, alloc_sizes, max_rows = 4000_000):
    mrc_profiler_instance = FootprintMRC(k=20_000_000)
    alloc_size_to_id = {alloc_size:index for index, alloc_size in enumerate(alloc_sizes)}
    class_id_to_allocs_per_slab_map = {index: 4 * 1024 * 1024 // alloc_size  for index, alloc_size in enumerate(alloc_sizes)}
    
    with open(trace_file_path, mode='r', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row_index, row in enumerate(reader):
            if row_index >= max_rows:
                break
            object_id = row['object_id']
            object_size = int(row['object_size'])
            object_size = max(24, object_size)
            object_size += (32 + len(str(object_id)))
            index = bisect.bisect_left(alloc_sizes, object_size)
            if index < len(alloc_sizes):
                alloc_size = alloc_sizes[index]
                mrc_profiler_instance.feed(object_id, alloc_size_to_id[alloc_size])
                
    
    return mrc_profiler_instance.solve_slab_reallocation(class_id_to_allocs_per_slab_map, {0: 0, 1:128, 2:0, 3:0, 4:0})



In [10]:
mr_old, mr_new, optimal_allocation, reassignment_plan, access_frequencies = process_file('/mydata/hongshu/traces/synth_static_202.csv', [256, 512, 1024, 2048, 4096])

MRC points for class_id 0: 
  0 slab: 1.0000
  1 slab: 0.9886
  2 slab: 0.9773
  3 slab: 0.9665
  4 slab: 0.9560
  10 slab: 0.8985
  20 slab: 0.8256
  40 slab: 0.7721
  50 slab: 0.7721
  60 slab: 0.7721
  80 slab: 0.7721
  100 slab: 0.7721
  120 slab: 0.7721
  128 slab: 0.7721
MRC points for class_id 1: 
  0 slab: 1.0000
  1 slab: 0.6942
  2 slab: 0.6320
  3 slab: 0.5914
  4 slab: 0.5609
  10 slab: 0.4563
  20 slab: 0.3808
  40 slab: 0.3511
  50 slab: 0.3511
  60 slab: 0.3511
  80 slab: 0.3511
  100 slab: 0.3511
  120 slab: 0.3511
  128 slab: 0.3511
MRC points for class_id 2: 
  0 slab: 1.0000
  1 slab: 0.9886
  2 slab: 0.9771
  3 slab: 0.9659
  4 slab: 0.9547
  10 slab: 0.8897
  20 slab: 0.7887
  40 slab: 0.6156
  50 slab: 0.5452
  60 slab: 0.4876
  80 slab: 0.4241
  100 slab: 0.4229
  120 slab: 0.4229
  128 slab: 0.4229
MRC points for class_id 3: 
  0 slab: 1.0000
  1 slab: 0.7165
  2 slab: 0.6514
  3 slab: 0.6082
  4 slab: 0.5758
  10 slab: 0.4568
  20 slab: 0.3528
  40 slab: 0.2441